<a href="https://colab.research.google.com/github/Prasanna-Mahajan-2006/DeepFake-Image-Classifier/blob/main/DSP_CNN_Term_Paper_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub

In [2]:
import os
import random
import shutil
import kagglehub
from google.colab import userdata
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import tensorflow_hub as hub

In [3]:
# Download the latest version of the OpenForensics dataset
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")

print("Path to dataset files:", path)

100%|██████████| 1.68G/1.68G [00:42<00:00, 42.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/manjilkarki/deepfake-and-real-images/versions/1


In [6]:
# 1. Pull credentials securely from Colab Secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. Download and Unzip
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d manjilkarki/deepfake-and-real-images

print("Unzipping dataset...")
!unzip -q deepfake-and-real-images.zip -d original_dataset
print("Unzip complete!")

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:42<00:00, 42.7MB/s]

Unzipping dataset...
Unzip complete!


In [4]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Peek inside the downloaded directory to confirm the folder name (usually 'Dataset')
print("Contents of the downloaded path:", os.listdir(path))



Contents of the downloaded path: ['Dataset']


In [5]:
# Extracting the datasets into variables
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 140002 files belonging to 2 classes.
Found 39428 files belonging to 2 classes.
Found 10905 files belonging to 2 classes.


In [ ]:
data_augmentation = tf.keras.Sequential([
    # Variances are standard deviations squared (0.229^2, 0.224^2, 0.225^2)
    # these are the actual mean intensities of red, blue and green from the ImageNet dataset.
    # Needed for normalization to run the gradient descent
    layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.0524, 0.0501, 0.0506]),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=10./360.),
    layers.RandomBrightness(factor=0.2), # Simulating color jittering
    layers.RandomContrast(factor=0.2)    # Simulating color jittering
])

# --- MODEL ARCHITECTURE ---
# Note: Using MobileNetV3Small as a placeholder since V4 is not native to Keras yet.
# By fine-tuning the model to detect subtle artifacts unique to deepfakes and employing optimization techniques, this work aims to deliver an effective yet scalable detection framework.
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = True

# The extracted features are refined through dimensionality reduction before being passed to a dense layer.
# To enhance generalization and minimize overfitting, techniques like dropout or normalization are applied before the dense layer.
# Finally, a binary classifier with a sigmoid activation function ensures accurate classification of the input.
model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# --- TRAINING ---
# The optimizer employed back-propagation and the Adam algorithm.
# The model was trained using a cross-entropy loss function.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# The MobileNetV4-Small model was trained over 11 epochs.
# Abhi: I set epochs = 3 for quicker results.
#       Also I added the steps_per_epoch to skip over some images, as the dataset is large.
history = model.fit(
    train_ds,
    steps_per_epoch = 20, # means that it trains only 20 batches per epoch.
    validation_data=val_ds,
    epochs=10
)

# --- EVALUATION ---
# During testing, the model's accuracy on the test data was calculated similarly to the training phase.
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
# 1. Configuration
BATCH_SIZE = 64
IMG_SIZE = (224, 224)
EPOCHS = 50
# Updated to point to your newly extracted 100k dataset
DATASET_DIR = '/content/my_100k_dataset'

# 2. Data Loading
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Optimize data loading for GPU
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

# 3. Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 4. Base Model (Native Keras MobileNetV3 Small)
base_model = keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze base layers

# 5. Build the Final Model
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)

# Pass through the base model (MobileNetV3 handles its own pixel scaling internally!)
x = base_model(x)

# Convert 3D feature maps to 1D feature vectors
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

# 6. Compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# 7. Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint("best_deepfake_model.keras", save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')
]

# 8. Training
print(f"Starting training for {EPOCHS} epochs...")
history = model.fit(
    train_dataset,
    epochs=EPOCH,
    validation_data=val_dataset,
    callbacks=callbacks
)

print("Training complete! Model saved as 'best_deepfake_model.keras'")

In [ ]:
source_dir = '/content/original_dataset'
dest_dir = '/content/my_100k_dataset'

# Create the clean destination directories
os.makedirs(os.path.join(dest_dir, 'Real'), exist_ok=True)
os.makedirs(os.path.join(dest_dir, 'Fake'), exist_ok=True)

all_real = []
all_fake = []

# 1. Scan the original dataset and map all file paths
print("Scanning original dataset...")
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(root, file)
            # The folder structure dictates the class
            if 'real' in root.lower():
                all_real.append(full_path)
            elif 'fake' in root.lower():
                all_fake.append(full_path)

print(f"Found {len(all_real)} total Real images and {len(all_fake)} total Fake images.")

# 2. Randomly sample 50,000 from each (with a safety net)
target_size = 50000
real_sample_size = min(target_size, len(all_real))
fake_sample_size = min(target_size, len(all_fake))

real_sample = random.sample(all_real, real_sample_size)
fake_sample = random.sample(all_fake, fake_sample_size)

# 3. Copy the sampled images to your new clean directory
print(f"Copying {real_sample_size} Real images to new directory...")
for i, img_path in enumerate(real_sample):
    shutil.copy(img_path, os.path.join(dest_dir, 'Real', f"real_{i}.jpg"))

print(f"Copying {fake_sample_size} Fake images to new directory...")
for i, img_path in enumerate(fake_sample):
    shutil.copy(img_path, os.path.join(dest_dir, 'Fake', f"fake_{i}.jpg"))

print("Done! Your 100,000 image dataset is beautifully prepped.")